# Why did the drivers change between 2017–2019 and 2023–2025?

Run from the project folder after both periods have been modelled in `run_model`

In [9]:
import ast

import numpy as np
import pandas as pd
import rasterio

from brt_modeling import cell_center, fit_brt, get_raster_paths, rebalance_1to1, sample_rasters
from config import PERIODS

P1, P2 = "2017_2019", "2023_2025"
FEATS = list(get_raster_paths(P1))
BOXES = {"Argentina": ((-70, -50), (-55, -35)), "Peru": ((-95, -75), (-25, -5))}  # (lon, lat) ranges of the squid hotspots
LAT, LON = np.meshgrid(89.875 - 0.25 * np.arange(720), -179.875 + 0.25 * np.arange(1440), indexing="ij")  # 0.25° cell centers


def load_events(p):
    """Events as in the main notebook, located where AIS was switched off, with a flag for carrier loitering in their cell."""
    df = pd.read_csv(f"data/processed/gaps_{p}_filtered.csv")
    start, end = (pd.to_datetime(df[c], utc=True) for c in ("start", "end"))
    t0, t1 = (pd.Timestamp(d, tz="UTC") for d in PERIODS[p])
    df = df[(start >= t0) & (end < t1 + pd.Timedelta(days=1))].drop_duplicates("id")
    gap = df["gap"].apply(ast.literal_eval)
    ev = pd.DataFrame({"lon": gap.str["off_position"].str["lon"], "lat": gap.str["off_position"].str["lat"],
                       "gear": df["gear_type"].str.upper(), "flag": df["vessel"].apply(lambda v: ast.literal_eval(v).get("flag")),
                       "vessel": df["vessel_id"], "reception": gap.str["positions_per_day_sat_reception"]})
    with rasterio.open(f"data/processed/loitering_hours_{p}.tif") as src:
        ev["loitering"] = np.array([v[0] for v in src.sample(zip(ev["lon"], ev["lat"]))], dtype=float) > 0
    return ev


EV = {p: load_events(p) for p in (P1, P2)}
MD = {p: pd.read_csv(f"model_results/{p}/model_df_full.csv") for p in (P1, P2)}

/tmp/ipykernel_63502/4067941536.py:21: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  df = df[(start >= t0) & (end < t1 + pd.Timedelta(days=1))].drop_duplicates("id")
/tmp/ipykernel_63502/4067941536.py:21: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  df = df[(start >= t0) & (end < t1 + pd.Timedelta(days=1))].drop_duplicates("id")


## The squid hotspots off Argentina and Peru

Number of events dropped, mostly those of Chinese-flagged vessels

In [10]:
rows = {}
for name, ((x0, x1), (y0, y1)) in BOXES.items():
    cells = (LON >= x0) & (LON < x1) & (LAT >= y0) & (LAT < y1)
    for p, ev in EV.items():
        e = ev[ev["lon"].between(x0, x1) & ev["lat"].between(y0, y1)]
        with rasterio.open(f"data/processed/loitering_hours_{p}.tif") as src:
            loitering = np.expm1(np.nan_to_num(src.read(1)))  # raster stores log(1 + hours)
        rows[(name, p)] = {"events": len(e), "Chinese-flagged events": (e["flag"] == "CHN").sum(),
                           "Chinese-flagged vessels": e.loc[e["flag"] == "CHN", "vessel"].nunique(),
                           "squid jigger events": (e["gear"] == "SQUID_JIGGER").sum(),
                           "median reception (positions/day)": e["reception"].median(), "carrier loitering (h)": loitering[cells].sum()}
pd.DataFrame(rows).round(0)

/tmp/ipykernel_63502/1566165757.py:7: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  loitering = np.expm1(np.nan_to_num(src.read(1)))  # raster stores log(1 + hours)
/tmp/ipykernel_63502/1566165757.py:7: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  loitering = np.expm1(np.nan_to_num(src.read(1)))  # raster stores log(1 + hours)
/tmp/ipykernel_63502/1566165757.py:7: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  loitering = np.expm1(np.nan_to_num(src.read(1)))  # raster stores log(1 + hours)
/tmp/ipykernel_63502/1566165757.py:7: DeprecationWarning: Setting the shape on a NumPy array has 

Argentina                Peru          
                                 2017_2019 2023_2025 2017_2019 2023_2025
events                              9549.0    2416.0    4081.0    1976.0
Chinese-flagged events              5088.0     655.0    2940.0    1067.0
Chinese-flagged vessels              442.0     211.0     543.0     351.0
squid jigger events                 3712.0     723.0    2751.0     825.0
median reception (positions/day)     387.0     371.0      68.0     317.0
carrier loitering (h)             105472.0  107707.0   93969.0  128473.0

## 2 Events in cells with carrier loitering (%)

In [11]:
pd.DataFrame({p: {"all": ev["loitering"].mean(), "squid jiggers": ev.loc[ev["gear"] == "SQUID_JIGGER", "loitering"].mean(),
                  "trawlers": ev.loc[ev["gear"] == "TRAWLERS", "loitering"].mean()} for p, ev in EV.items()}).mul(100).round(1)

,2017_2019,2023_2025
all,42.1,23.0
squid jiggers,85.0,69.0
trawlers,56.7,21.4


## Flags with the largest decrease and increase in events

Five flags each, ranked by the change in the number of events

In [15]:
change = EV[P2]["flag"].value_counts().sub(EV[P1]["flag"].value_counts(), fill_value=0).astype(int).sort_values()
pd.concat([change.head(5), change.tail(5)]).to_frame("change in events")

,change in events
flag,
CHN,-8039
TWN,-6655
KOR,-1965
ESP,-828
FRA,-639
BRA,1397
USA,1412
VEN,2256
LKA,2318


## SST variability at presences and absences (full models)

The events moved into more stable water (less SST variability).

In [13]:
for p, d in MD.items():
    a = d[d["presence"] == 0]
    tropical_pacific = a["lat"].between(-20, 23) & ((a["lon"] >= 125) | (a["lon"] <= -70))
    print(f"{p}: median SST SD {d.loc[d['presence'] == 1, 'sst_var'].median():.2f} °C at presences, {a['sst_var'].median():.2f} °C at absences; "
          f"absences in the tropical Pacific {tropical_pacific.mean():.1%}, in the NW Pacific {((a['lat'] >= 23) & (a['lon'] >= 100)).mean():.1%}")

2017_2019: median SST SD 2.08 °C at presences, 1.63 °C at absences; absences in the tropical Pacific 32.1%, in the NW Pacific 8.1%
2023_2025: median SST SD 1.62 °C at presences, 1.79 °C at absences; absences in the tropical Pacific 22.1%, in the NW Pacific 12.3%


## Swap test

Full model refitted with the presences of one period, the absences of another and the driver rasters of a third. The new events remove loitering, the new absences put SST SD on top.

In [6]:
def swap(events, absences, rasters):
    pres = MD[events].loc[MD[events]["presence"] == 1, ["lon", "lat"]]
    absn = MD[absences].loc[MD[absences]["presence"] == 0, ["lon", "lat"]]
    lon, lat = cell_center(EV[events]["lon"].values, EV[events]["lat"].values)
    event_cells = set(zip(lon.round(3), lat.round(3)))
    absn = absn[[c not in event_cells for c in zip(absn["lon"].round(3), absn["lat"].round(3))]]
    d = sample_rasters(pd.concat([pres.assign(presence=1), absn.assign(presence=0)], ignore_index=True), get_raster_paths(rasters))
    d["loitering_hours"] = d["loitering_hours"].fillna(0)
    model, _ = fit_brt(rebalance_1to1(d.dropna(subset=FEATS[1:])), FEATS, n_iter=1)
    return pd.Series(100 * model.feature_importances_.astype(float), index=FEATS)


combos = [(P1, P1, P1), (P1, P2, P1), (P2, P1, P1), (P2, P2, P1), (P1, P1, P2), (P2, P2, P2)]
pd.DataFrame({c: swap(*c) for c in combos}).T.rename_axis(["events", "absences", "rasters"]).round(1)

Explained deviance: 26.17%  AUC: 0.741 ± 0.000  TSS: 0.366 ± 0.000
Explained deviance: 30.57%  AUC: 0.762 ± 0.000  TSS: 0.400 ± 0.000
Explained deviance: 22.09%  AUC: 0.763 ± 0.000  TSS: 0.391 ± 0.000
Explained deviance: 19.76%  AUC: 0.731 ± 0.000  TSS: 0.350 ± 0.000
Explained deviance: 24.53%  AUC: 0.733 ± 0.000  TSS: 0.352 ± 0.000
Explained deviance: 20.55%  AUC: 0.736 ± 0.000  TSS: 0.359 ± 0.000


loitering_hours  dist2coast  dist_mpa  \
events    absences  rasters                                            
2017_2019 2017_2019 2017_2019             47.3        15.7       2.4   
          2023_2025 2017_2019             42.4        12.7       2.8   
2023_2025 2017_2019 2017_2019              8.9        25.0       3.3   
          2023_2025 2017_2019              8.0        16.4       3.8   
2017_2019 2017_2019 2023_2025             34.7        16.6       3.1   
2023_2025 2023_2025 2023_2025             11.1        15.6       2.2   

                               dist_piracy   chl   sst  sst_var  eke  
events    absences  rasters                                           
2017_2019 2017_2019 2017_2019          1.4   9.3  10.9      9.8  3.1  
          2023_2025 2017_2019          4.6  10.4  11.5     13.1  2.5  
2023_2025 2017_2019 2017_2019          1.5  14.5  24.9     17.0  4.9  
          2023_2025 2017_2019          4.8  13.1  21.2     29.3  3.3  
2017_2019 2017_2019 2023_2025          1.7  19.8   9.8      9.5  4.8  
2023_2025 2023_2025 2023_2025          6.6  15.7  17.7     29.0  2.2